# 02 — Zero-shot baselines

We need numbers to beat. This notebook runs two zero-shot baselines on the frozen 50-row hold-out and computes the first two columns of the eval table:

1. **Llama 3.2 3B Instruct** — base model, no fine-tuning, no LoRA.
2. **Gemini 2.5 Flash** — frontier baseline via Google AI Studio (free tier).

Predictions are saved to `reports/predictions/<name>.jsonl`. The bench in notebook 05 reads from these JSONL files so re-runs are cheap.

Requires: `GOOGLE_API_KEY` in `.env` for the frontier baseline. The base-model baseline requires a GPU and `pip install -e ".[train]"`; on CPU you can skip cell 3 and load the predictions from the committed `reports/predictions/base.jsonl` once Weekend 3 has populated them.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from earningslora.config import get_settings
from earningslora.data.ectsum import load_ectsum, make_holdout
from earningslora.evaluation.frontier_baseline import frontier_summary
from earningslora.evaluation.harness import load_predictions, run_holdout
from earningslora.evaluation.numeric_recall import numeric_recall
from earningslora.evaluation.rouge import rouge_scores

settings = get_settings()
predictions_dir = Path("../reports/predictions")
predictions_dir.mkdir(parents=True, exist_ok=True)
predictions_dir

## 1. Hold-out

Single source of truth across base / FT / frontier. Same seed, same indices.

In [ ]:
raw = load_ectsum()
holdout = make_holdout(raw)
print(f"Hold-out size: {len(holdout)} (seed={settings.eval_seed})")

## 2. Frontier baseline (Gemini 2.5 Flash)

All calls cached to SQLite — re-running this cell costs zero quota.
Skipped if `GOOGLE_API_KEY` isn't set in the environment.

In [ ]:
import os

if os.environ.get("GOOGLE_API_KEY"):
    frontier_path = run_holdout(
        name="frontier",
        summarise=frontier_summary,
        holdout=holdout,
        output_dir=predictions_dir,
    )
    print(f"Frontier predictions: {frontier_path}")
else:
    print("GOOGLE_API_KEY not set — skipping frontier baseline.")

## 3. Base-model baseline (Llama 3.2 3B Instruct, GPU required)

Lazy-imported below so the notebook can be skimmed on CPU. Lands in Weekend 3 alongside `inference/generate.py`.

In [ ]:
# from earningslora.inference.load import load_with_adapter  # weekend 3
# from earningslora.inference.generate import generate_summary  # weekend 3
#
# model, tokenizer = load_base(settings.base_model)
# def base_summarise(transcript: str) -> str:
#     return generate_summary(model, tokenizer, transcript)
#
# base_path = run_holdout(
#     name="base",
#     summarise=base_summarise,
#     holdout=holdout,
#     output_dir=predictions_dir,
# )
print("Base-model baseline lands in Weekend 3.")

## 4. First eval pass

ROUGE + numeric-recall on whatever predictions we have. The LLM-as-judge column comes in notebook 05 (it's a head-to-head comparison; we need at least two configurations).

In [ ]:
import statistics


def evaluate(predictions_path):
    rows = load_predictions(predictions_path)
    preds = [r["prediction"] for r in rows]
    refs = [r["reference"] for r in rows]
    transcripts = [r["transcript"] for r in rows]
    nr = [numeric_recall(t, p).recall for t, p in zip(transcripts, preds)]
    rouge = rouge_scores(preds, refs)
    latencies = [r["latency_ms"] for r in rows]
    return {
        **rouge,
        "numeric_recall": round(statistics.mean(nr), 4) if nr else 0.0,
        "latency_ms_p50": round(statistics.median(latencies), 1) if latencies else 0.0,
    }


for name in ["frontier", "base", "ft"]:
    p = predictions_dir / f"{name}.jsonl"
    if p.exists():
        print(f"{name:10s}: {evaluate(p)}")
    else:
        print(f"{name:10s}: predictions not yet generated")

## Next

Notebook 03 trains the QLoRA adapter on Kaggle/Colab T4. It expects this notebook's frontier predictions to already be on disk so re-runs of the bench don't re-spend quota.